# House Price Prediction with Linear Regression
**OIBSIP Data Analytics — Level 2, Task 1**

This is a learning project. The included dataset is synthetic demo data, not real market data.

## 1. Import libraries and load data

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE = Path.cwd()
if not (BASE / "data/house_prices.csv").exists() and (BASE / "House_Price_Prediction/data/house_prices.csv").exists():
    BASE = BASE / "House_Price_Prediction"
df = pd.read_csv(BASE / "data/house_prices.csv")
df.head()


## 2. Inspect and explore the dataset

In [ ]:
print("Shape:", df.shape)
display(df.isna().sum().to_frame("Missing values"))
display(df.describe(include="all"))
plt.figure(figsize=(8,5))
plt.scatter(df["Area_sqft"], df["Price_INR"], alpha=.6)
plt.xlabel("Area (sq ft)"); plt.ylabel("Price (INR)")
plt.title("House Area vs Price"); plt.show()


## 3. Correlation heatmap

In [ ]:
numeric = df.select_dtypes(include="number")
plt.figure(figsize=(8,6))
plt.imshow(numeric.corr(), aspect="auto")
plt.xticks(range(len(numeric.columns)), numeric.columns, rotation=45, ha="right")
plt.yticks(range(len(numeric.columns)), numeric.columns)
plt.colorbar(label="Correlation")
plt.title("Correlation Heatmap"); plt.tight_layout(); plt.show()


## 4. Prepare features and split data
The `Location` column is categorical, so OneHotEncoder converts it into numeric indicator columns. The split holds out 20% of rows for testing.

In [ ]:
target = "Price_INR"
features = ["Area_sqft", "Bedrooms", "Age_years", "Location", "Quality_Rating"]
X, y = df[features], df[target]
numeric_features = ["Area_sqft", "Bedrooms", "Age_years", "Quality_Rating"]
categorical_features = ["Location"]
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 5. Train and compare models

In [ ]:
models = {"Linear Regression": LinearRegression(), "Ridge Regression": Ridge(alpha=1.0)}
results, fitted = [], {}
for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({"Model": name, "MAE_INR": mean_absolute_error(y_test, pred),
                    "RMSE_INR": mean_squared_error(y_test, pred)**0.5,
                    "R2": r2_score(y_test, pred)})
    fitted[name] = (pipe, pred)
metrics = pd.DataFrame(results).round(3)
display(metrics)
best_name = metrics.sort_values("RMSE_INR").iloc[0]["Model"]
best_pipe, best_pred = fitted[best_name]
print("Best model by RMSE:", best_name)


## 6. Actual vs predicted and residual plots

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(y_test, best_pred, alpha=.7)
low, high = min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual Price (INR)"); plt.ylabel("Predicted Price (INR)")
plt.title(f"Actual vs Predicted — {best_name}"); plt.show()

residuals = y_test - best_pred
plt.figure(figsize=(7,5))
plt.scatter(best_pred, residuals, alpha=.7)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Price (INR)"); plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot"); plt.show()


## 7. Interpret model coefficients

In [ ]:
feature_names = best_pipe.named_steps["preprocessor"].get_feature_names_out()
coef = best_pipe.named_steps["model"].coef_
coef_df = pd.DataFrame({"Feature": feature_names, "Coefficient_INR": coef})
display(coef_df.reindex(coef_df["Coefficient_INR"].abs().sort_values(ascending=False).index))


## 8. Conclusion
- Regression estimates a continuous value, here house price.
- MAE and RMSE show average prediction error in INR; R² summarises variance explained on the test set.
- Coefficients describe the model's learned associations, not necessarily causal effects.

**Dataset disclosure:** Synthetic practice data only. Replace it with a public real-world housing dataset if the task evaluator requires one. Do not use this demo model to make actual property decisions.